# Potato Leaf Disease Classification

This notebook trains a ResNet50 transfer-learning model to classify potato leaf images as early blight, late blight, or healthy. Place the dataset under `../data/PotatoPlants` when running from this notebook folder, or `data/PotatoPlants` when running from the repository root. Model checkpoints are generated locally under `outputs/` and are intentionally not committed to GitHub.


In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision import datasets, models
import os
import numpy as np
import matplotlib.pyplot as plt

In [ ]:

ttransform = transforms.Compose([
    transforms.RandomResizedCrop(512),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split


data_dir = DATA_DIR
full_dataset = datasets.ImageFolder(root=data_dir, transform=ttransform)

#split
total_size = len(full_dataset)
train_size = int(0.8 * total_size)
val_size   = int(0.1 * total_size)
test_size  = total_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

In [ ]:

# ResNet50 initialized with ImageNet weights when available.
try:
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
except Exception as exc:
    print(f"Could not load ImageNet weights automatically: {exc}")
    model = models.resnet50(weights=None)



num_classes = len(full_dataset.classes)  

# freezeeeeeeeeee
for param in model.parameters():
    param.requires_grad = False
# refine the fully connected layer
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_classes)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=1e-4)  

In [ ]:
num_classes

In [ ]:


def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=20):
    best_acc = 0.0
    for epoch in range(num_epochs):
       
        model.train()
        running_loss = 0.0
        correct_preds = 0
        total_preds   = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

           
            running_loss   += loss.item() * inputs.size(0)
            _, preds       = outputs.max(1)
            correct_preds  += (preds == labels).sum().item()
            total_preds    += labels.size(0)

        epoch_train_loss = running_loss / len(train_loader.dataset)
        epoch_train_acc  = correct_preds   / total_preds * 100

        
        model.eval()
        running_val_loss = 0.0
        val_corrects     = 0
        val_totals       = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss    = criterion(outputs, labels)

               
                running_val_loss += loss.item() * inputs.size(0)
                _, preds         = outputs.max(1)
                val_corrects    += (preds == labels).sum().item()
                val_totals      += labels.size(0)

        epoch_val_loss = running_val_loss / len(val_loader.dataset)
        epoch_val_acc  = val_corrects     / val_totals * 100

        print(f"Epoch {epoch+1}/{num_epochs}  "
              f"Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.2f}%  "
              f"Val   Loss: {epoch_val_loss:.4f}, Val   Acc: {epoch_val_acc:.2f}%")

        
        if epoch_val_acc > best_acc:
            best_acc = epoch_val_acc
            torch.save(model.state_dict(), OUTPUT_DIR / 'best_model.pth')

    print(f"Done. Best Val acc：{best_acc:.2f}%")



train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=20)

In [ ]:

model.load_state_dict(torch.load(OUTPUT_DIR / 'best_model.pth', map_location=device))

train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10)

In [ ]:
torch.save(model.state_dict(), OUTPUT_DIR / 'last_model.pth')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker 
train_losses = [0.8440, 0.6949, 0.6152, 0.5510, 0.4976, 0.4686, 0.4302, 0.4034, 0.3824, 0.3729, 0.3526, 0.3324, 0.3194, 0.3181, 0.2935, 0.2930, 0.2723, 0.2741, 0.2728, 0.2512]
val_losses = [0.7672, 0.6597, 0.5701, 0.5140, 0.4858, 0.4543, 0.4090, 0.4110, 0.3494, 0.3470, 0.3095, 0.2893, 0.2735, 0.2782, 0.2659, 0.2319, 0.2302, 0.2428, 0.2405, 0.2170]
epochs = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

plt.figure(figsize=(10, 6))
plt.plot(epochs, train_losses, label='Train Loss', marker='o', linestyle='-', color='blue')
plt.plot(epochs, val_losses, label='Val Loss', marker='s', linestyle='--', color='orange')
plt.title('Training and Validation Loss',fontsize= 18)
plt.xlabel('Epoch',fontsize= 18)
plt.ylabel('Loss',fontsize= 18)
plt.legend(fontsize= 12)

plt.gca().xaxis.set_major_formatter(ticker.FormatStrFormatter('%d'))

plt.grid(True)
plt.savefig(ASSET_DIR / 'loss.png', dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker  

train_acc = [69.73, 84.49, 84.66, 85.94, 86.93, 86.40, 87.62, 89.08, 89.42, 88.61, 90.01, 90.70, 91.57, 91.05, 91.75, 92.45, 92.68, 91.87, 92.50, 93.67]
val_acc = [81.86, 84.19, 85.58, 85.12, 86.05, 86.98, 86.98, 87.91, 90.70, 90.23, 90.70, 93.02, 93.95, 94.42, 93.02, 95.35, 94.88, 94.88, 92.09, 95.35]
epochs = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]

plt.figure(figsize=(10, 6))
plt.plot(epochs, train_acc, label='Train Acc', marker='o', linestyle='-', color='blue')
plt.plot(epochs, val_acc, label='Val Acc', marker='s', linestyle='--', color='orange')
plt.title('Training and Validation Accuracy(%)',fontsize= 18)
plt.xlabel('Epoch',fontsize= 18)
plt.ylabel('Accuracy',fontsize= 18)
plt.legend(fontsize= 12)


plt.gca().xaxis.set_major_formatter(ticker.FormatStrFormatter('%d'))

plt.grid(True)
plt.savefig(ASSET_DIR / 'accuracy.png', dpi=300)
plt.show()

In [ ]:
# misclassified eg

def collect_misclassified(model, full_dataset, val_ds, val_loader, preprocess, device):
    model.eval()
    misclassified = []
    with torch.no_grad():
        for batch_idx, (inputs, labels) in enumerate(val_loader):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            preds   = outputs.argmax(dim=1)

           
            wrongs = (preds != labels).nonzero(as_tuple=False).flatten().tolist()
            for w in wrongs:
                
                global_idx = val_ds.indices[batch_idx * val_loader.batch_size + w]
                img_path, true_label = full_dataset.samples[global_idx]
                pred_label = preds[w].item()
                misclassified.append((img_path, full_dataset.classes[true_label], full_dataset.classes[pred_label]))
    return misclassified


mis = collect_misclassified(model, full_dataset, val_dataset, val_loader, ttransform, device)

print(f"Last epoch misclassified: {len(mis)} images")
for path, true_cls, pred_cls in mis:
    print(f"File: {path}  True: {true_cls}  Pred: {pred_cls}")



In [ ]:


def evaluate_model(model, test_loader):
    model.load_state_dict(torch.load(OUTPUT_DIR / 'last_model.pth', map_location=device))
    model.eval()
    correct_preds = 0
    total_preds = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            correct_preds += torch.sum(preds == labels).item()
            total_preds += labels.size(0)
    
    accuracy = correct_preds / total_preds * 100
    print(f"Test Accuracy: {accuracy:.2f}%")


evaluate_model(model, test_loader)

In [ ]:
## Grad-CAM

import torch
import torch.nn.functional as F
import cv2
import numpy as np
import matplotlib.pyplot as plt
from captum.attr import LayerGradCam
from torchvision import transforms, models
from PIL import Image
import torch
from torchvision import models
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet50(pretrained=False)
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, 3)


model = model.to(device)
model.load_state_dict(torch.load(OUTPUT_DIR / "best_model.pth", map_location=device))
model.eval()


activations = None
gradients = None

def forward_hook(module, inp, out):
    global activations
    activations = out.detach()

def backward_hook(module, grad_in, grad_out):
    global gradients
    gradients = grad_out[0].detach()

target_layer = model.layer4[-1].conv3
target_layer.register_forward_hook(forward_hook)
target_layer.register_backward_hook(backward_hook)


preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])


img_path = DATA_DIR / 'Potato___Late_blight' / 'e26ad557-11c8-44fd-aad1-dea51c613742___RS_LB 5115.JPG'
orig_img = Image.open(img_path).convert("RGB")
input_tensor = preprocess(orig_img).unsqueeze(0).to(device)


model.zero_grad()
output = model(input_tensor)
pred_class = output.argmax(dim=1).item()
output[0, pred_class].backward()


weights = torch.mean(gradients, dim=(2,3), keepdim=True)     
cam = torch.sum(weights * activations, dim=1, keepdim=True)   
cam = F.relu(cam)                                             

cam = F.interpolate(cam, size=(224,224), mode='bilinear', align_corners=False)
cam = cam.squeeze(0).squeeze(0).cpu().numpy()


cam -= cam.min()
cam /= cam.max()


heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
overlay = np.array(orig_img.resize((224,224))) * 0.5 + heatmap * 0.5
overlay = overlay.astype(np.uint8)


fig, axes = plt.subplots(1, 3, figsize=(12,4))
axes[0].imshow(orig_img); axes[0].set_title("原始图像");       axes[0].axis('off')
axes[1].imshow(cam, cmap='jet'); axes[1].set_title("Grad-CAM"); axes[1].axis('off')
axes[2].imshow(overlay); axes[2].set_title(f"Overlay (Class {pred_class})"); axes[2].axis('off')
plt.tight_layout()

plt.savefig('misclassif1.png')
plt.show()
